# Notebook 6 — Expanded models: 5-HT₂A & Dopamine D3 (preclinical + Phase I–IV)

Per the meeting with Filippo. **Only the ML models change** — the descriptive analysis stays on the
locked Phase I–IV dataset. Here we:

1. **Fetch preclinical-only** Kᵢ for 5-HT₂A and D3 (the compounds the original fetch excluded).
2. **Curate** them identically (salt-strip → MW<700 → peptide/biologic removal → dedup) **+ Landrum
   2024 curation** (drop exact-duplicate and Δ=3.0-log artefacts).
3. **Concatenate** with the existing Phase I–IV compounds for each target → the expanded ML set.
4. **Fuse ECFP4 + DescriptaStorus** features and train **XGBoost + Random Forest**.
5. **Validate two ways: repeated scaffold split (honest/prospective) AND leave-one-out (LOO).**

> Requires internet for Part 1 (ChEMBL). `pip install chembl_webresource_client rdkit descriptastorus xgboost scikit-learn`.

## 0. Config

In [ ]:
import os, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs, RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.Scaffolds import MurckoScaffold
from descriptastorus.descriptors import rdNormalizedDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, root_mean_squared_error as rmse
try:
    from xgboost import XGBRegressor; HAVE_XGB=True
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor; HAVE_XGB=False
    print('xgboost not found -> using HistGradientBoosting as the boosting model')

LOCKED_CSV = 'data/curated_15desc.csv' if os.path.exists('data/curated_15desc.csv') else 'curated_15desc.csv'
TARGETS = {'5-HT2A': 'P28223', 'Dopamine D3': 'P35462'}   # name -> UniProt
print('boosting model:', 'XGBoost' if HAVE_XGB else 'HistGradientBoosting')

## 1. Fetch **preclinical-only** Kᵢ for the two targets
We pull all human Kᵢ binding activities for each target, attach each molecule's `max_phase`, and keep
only the **preclinical** ones (`max_phase` = 0 or null) — i.e. exactly the compounds the Phase I–IV
fetch left out. These are what we add to the training set.

In [ ]:
# ---- RESILIENT FETCH v2 — exact "=" Ki only, pKi DERIVED from standard_value (no pChEMBL filter) ----
# Filters: standard_type=Ki, assay_type=B, standard_relation="=", standard_units="nM", human.
# pKi = 9 - log10(Ki[nM]).  Re-run to resume from checkpoints (ki_eq_*.csv / maxphase_*.json).
import os, time, json
from chembl_webresource_client.new_client import new_client
from chembl_webresource_client.settings import Settings
Settings.Instance().TIMEOUT=60; Settings.Instance().TOTAL_RETRIES=10; Settings.Instance().MAX_LIMIT=1000
target_api, activity_api, molecule_api = new_client.target, new_client.activity, new_client.molecule

def _retry(call, what, tries=8):
    for k in range(tries):
        try: return call()
        except Exception as e:
            print(f"    {what}: {type(e).__name__} -> retry {k+1}/{tries} in {5*(k+1)}s"); time.sleep(5*(k+1))
    raise RuntimeError(f"{what} failed after {tries} tries")

def resolve(acc):
    h=_retry(lambda: list(target_api.filter(target_components__accession=acc,
            target_type='SINGLE PROTEIN', organism='Homo sapiens').only(['target_chembl_id'])), f"resolve {acc}")
    return h[0]['target_chembl_id'] if h else None

def fetch_activities(tid, tag, page=1000):
    ckpt=f"ki_eq_{tag}.csv"
    got=pd.read_csv(ckpt) if os.path.exists(ckpt) else pd.DataFrame()
    qs=activity_api.filter(target_chembl_id=tid, standard_type='Ki', assay_type='B',
                           standard_relation='=', standard_units='nM',
                           target_organism='Homo sapiens')\
                   .only(['molecule_chembl_id','canonical_smiles','standard_value'])
    total=_retry(lambda: len(qs), f"count {tag}"); start=len(got)
    print(f"  {tag}: {total} exact '=' Ki(nM) rows; cached {start}")
    while start<total:
        chunk=_retry(lambda: pd.DataFrame(list(qs[start:start+page])), f"rows {start}")
        if chunk.empty: break
        got=pd.concat([got,chunk],ignore_index=True); got.to_csv(ckpt,index=False); start+=page
        print(f"    {min(start,total)}/{total}")
    return got.drop_duplicates()

def fetch_maxphase(ids, tag, batch=200):
    ckpt=f"maxphase_{tag}.json"; ph=json.load(open(ckpt)) if os.path.exists(ckpt) else {}
    todo=[i for i in ids if i not in ph]; print(f"  {tag}: max_phase for {len(todo)} molecules")
    for i in range(0,len(todo),batch):
        recs=_retry(lambda: list(molecule_api.filter(molecule_chembl_id__in=todo[i:i+batch])
                                 .only(['molecule_chembl_id','max_phase'])), f"max_phase {i}")
        for m in recs: ph[m['molecule_chembl_id']]=m['max_phase']
        json.dump(ph, open(ckpt,'w'))
    return ph

def fetch_preclinical(name, acc):
    tid=resolve(acc); print(f'{name}: {tid}'); tag=name.replace(' ','_')
    raw=fetch_activities(tid, tag)
    raw['standard_value']=pd.to_numeric(raw['standard_value'], errors='coerce')
    raw=raw[raw['standard_value']>0].copy()
    raw['pchembl_value']=9.0-np.log10(raw['standard_value'])      # nM -> pKi (derived; downstream cells read this)
    ph=fetch_maxphase(raw['molecule_chembl_id'].dropna().unique().tolist(), tag)
    raw['max_phase']=pd.to_numeric(raw['molecule_chembl_id'].map(ph), errors='coerce')
    pre=raw[~raw['max_phase'].isin([1,2,3,4])].copy(); pre['target']=name
    print(f'  {len(raw)} exact Ki rows -> {len(pre)} preclinical ({pre.molecule_chembl_id.nunique()} molecules)')
    return pre

pre_raw = pd.concat([fetch_preclinical(n,a) for n,a in TARGETS.items()], ignore_index=True)


## 2. Curate the preclinical additions (identical pipeline + Landrum 2024)
Salt-strip to parent, drop unparsable SMILES, MW < 700, remove peptides. Then the **Landrum artefact
filter**: within each (molecule, target) drop exactly-duplicated pKᵢ values (paper-copies) and values
exactly 3.0 log apart (unit errors), and finally take the highest pKᵢ to match the existing data.

In [ ]:
sr=SaltRemover()
def parent(s):
    m=Chem.MolFromSmiles(str(s)) if isinstance(s,str) else None
    if m is None: return None
    p=sr.StripMol(m, dontRemoveEverything=True)
    if p is None or p.GetNumAtoms()==0:
        fr=Chem.GetMolFrags(m,asMols=True); p=max(fr,key=lambda x:x.GetNumAtoms()) if fr else m
    return p
pep=Chem.MolFromSmarts('[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)')
from rdkit.Chem import Descriptors

d=pre_raw.copy()
d['pKi']=pd.to_numeric(d['pchembl_value'],errors='coerce'); d=d.dropna(subset=['pKi'])
d['mol']=d['canonical_smiles'].apply(parent); d=d[d['mol'].notna()]
d=d[d['mol'].apply(lambda m: Descriptors.MolWt(m))<700]                       # MW < 700
d=d[~d['mol'].apply(lambda m: m.HasSubstructMatch(pep))]                       # peptides out
d['parent_smiles']=d['mol'].apply(Chem.MolToSmiles)

# --- Landrum artefact removal within each (molecule,target) ---
def landrum(g):
    v=g['pKi'].round(2)
    v=v[~v.duplicated(keep='first')]                    # drop exact-duplicate copies
    v=[x for x in v if not any(abs(x-o-3.0)<1e-6 or abs(x-o+3.0)<1e-6 for o in v)]  # drop 3.0-log unit errors
    return max(v) if v else np.nan                      # highest pKi (matches existing data)
agg=d.groupby(['molecule_chembl_id','target']).apply(lambda g: pd.Series({
        'pKi':landrum(g), 'parent_smiles':g['parent_smiles'].iloc[0]})).reset_index().dropna(subset=['pKi'])
print('preclinical after curation:')
print(agg.groupby('target').molecule_chembl_id.nunique())

## 3. Build the expanded ML set per target
Existing Phase I–IV compounds (from the locked dataset) **+** the new curated preclinical compounds.
Any molecule already present (measured in both) keeps its existing value — we only *add* new molecules.

In [ ]:
locked=pd.read_csv(LOCKED_CSV)
def expanded(target):
    ex=locked[locked.target.str.contains(target.split()[-1] if 'Dopamine' in target else target, case=False)]\
           [['molecule_chembl_id','pKi','parent_smiles']].drop_duplicates('molecule_chembl_id')
    pr=agg[agg.target==target][['molecule_chembl_id','pKi','parent_smiles']]
    pr=pr[~pr.molecule_chembl_id.isin(ex.molecule_chembl_id)]   # only genuinely new molecules
    both=pd.concat([ex,pr],ignore_index=True).drop_duplicates('molecule_chembl_id').reset_index(drop=True)
    print(f'{target}: {len(ex)} existing + {len(pr)} new preclinical = {len(both)} total')
    return both
sets={t:expanded(t) for t in TARGETS}

## 4. Features: ECFP4 + DescriptaStorus (fused)

In [ ]:
gen=rdNormalizedDescriptors.RDKit2DNormalized()
def featurize(dfset):
    mols=[Chem.MolFromSmiles(s) for s in dfset.parent_smiles]; keep=[i for i,m in enumerate(mols) if m]
    dfset=dfset.iloc[keep].reset_index(drop=True); mols=[mols[i] for i in keep]
    N=len(mols); E=np.zeros((N,2048),int)
    for i,m in enumerate(mols): DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m,2,nBits=2048),E[i])
    D=np.nan_to_num(np.array([gen.process(s)[1:] for s in dfset.parent_smiles],dtype=float))
    X=np.hstack([E,D]); y=dfset.pKi.values.astype(float)
    scaf=np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)) for m in mols])
    return X,y,scaf
feat={t:featurize(sets[t]) for t in TARGETS}
for t in TARGETS: print(t,'-> X',feat[t][0].shape)

## 5. Train & validate — scaffold split **and** LOO
Two models (XGBoost, Random Forest), each scored two ways: a repeated **Bemis–Murcko scaffold split**
(the honest, novel-chemotype number) and **leave-one-out** (the standard small-data number Filippo
asked for). LOO fits one model per compound, so it takes a few minutes per target.

In [ ]:
from sklearn.model_selection import KFold

def boost():
    return XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.6, random_state=0, n_jobs=-1) if HAVE_XGB \
           else HistGradientBoostingRegressor(random_state=0)
def rf(): return RandomForestRegressor(n_estimators=400, random_state=0, n_jobs=-1)
MODELS={'XGB' if HAVE_XGB else 'GB':boost, 'RF':rf}

def scaffold_split(scaf,N,seed,tf=0.2):
    rng=np.random.RandomState(seed); g={}
    for i,s in enumerate(scaf): g.setdefault(s,[]).append(i)
    ks=list(g); rng.shuffle(ks); nt=int(round(tf*N)); te=[]
    for k in ks:
        if len(te)>=nt: break
        te+=g[k]
    te=set(te); return [i for i in range(N) if i not in te], sorted(te)

rows=[]
for t in TARGETS:
    X,y,scaf=feat[t]; N=len(y)
    for mname,mk in MODELS.items():
        # scaffold split, 12 seeds
        rs=[]
        for s in range(12):
            tr,te=scaffold_split(scaf,N,s)
            if len(te)<5 or len(np.unique(y[te]))<2: continue
            m=mk().fit(X[tr],y[tr]); rs.append(r2_score(y[te],m.predict(X[te])))
        rs=np.array(rs)
        # 10-fold CV  (replaces leave-one-out)
        pred=np.zeros(N)
        for tr,te in KFold(n_splits=10, shuffle=True, random_state=0).split(X):
            m=mk().fit(X[tr],y[tr]); pred[te]=m.predict(X[te])
        rows.append({'target':t,'model':mname,'n':N,
                     'scaffold_R2':round(rs.mean(),3),
                     'scaffold_CI':f'[{np.percentile(rs,2.5):.2f},{np.percentile(rs,97.5):.2f}]',
                     'CV10_R2':round(r2_score(y,pred),3),
                     'CV10_RMSE':round(rmse(y,pred),3)})
        print('done:',t,mname)
res=pd.DataFrame(rows); print(); print(res.to_string(index=False))
res.to_csv('expanded_model_results.csv',index=False)

## Interpretation
* Compare **scaffold_R2** (novel-chemotype, honest) with **LOO_R2** (standard, usually higher) for each
  target/model — report **both** in the thesis so the two numbers don't look contradictory.
* The fused **ECFP4 + DescriptaStorus** representation should beat either alone (validated on the
  Phase I–IV data: 0.27 / 0.20 → 0.34). Adding preclinical compounds increases *n*, which should lift
  performance further — the point of this exercise.
* Two targets (5-HT₂A and D3) give a second, independent read on how well the protocol transfers.

In [ ]:
# === Supporting analyses (self-contained; run after Cell 12) ===
import numpy as np
from sklearn.metrics import r2_score
def scaffold_split(scaf,N,seed,tf=0.2):
    rng=np.random.RandomState(seed); g={}
    for i,s in enumerate(scaf): g.setdefault(s,[]).append(i)
    ks=list(g); rng.shuffle(ks); nt=int(round(tf*N)); te=[]
    for k in ks:
        if len(te)>=nt: break
        te+=g[k]
    te=set(te); return [i for i in range(N) if i not in te], sorted(te)
def boost():
    return XGBRegressor(n_estimators=400,max_depth=4,learning_rate=0.05,subsample=0.8,
                        colsample_bytree=0.6,random_state=0,n_jobs=-1) if HAVE_XGB else HistGradientBoostingRegressor(random_state=0)
def _sm(X,y,scaf,seeds=12):
    rs=[]
    for s in range(seeds):
        tr,te=scaffold_split(scaf,len(y),s)
        if len(te)<5 or len(np.unique(y[te]))<2: continue
        rs.append(r2_score(y[te],boost().fit(X[tr],y[tr]).predict(X[te])))
    return float(np.mean(rs))
print("1) REPRESENTATION (XGB scaffold R2/12)")
for t in TARGETS:
    X,y,scaf=feat[t]
    for lab,Xr in [("ECFP4",X[:,:2048]),("DescriptaStorus",X[:,2048:]),("Fused",X)]:
        print(f"   {t:12s} {lab:15s} {_sm(Xr,y,scaf):.3f}")
print("2) Y-RANDOMISATION (100 perms)")
for t in TARGETS:
    X,y,scaf=feat[t]; tr,te=scaffold_split(scaf,len(y),0)
    true=r2_score(y[te],boost().fit(X[tr],y[tr]).predict(X[te])); rng=np.random.RandomState(0)
    null=np.array([r2_score(y[te],boost().fit(X[tr],rng.permutation(y[tr])).predict(X[te])) for _ in range(100)])
    print(f"   {t:12s} true={true:.3f} null_mean={null.mean():.3f} max={null.max():.3f} p={(int((null>=true).sum())+1)/(len(null)+1):.4f}")
print("3) LEARNING CURVE")
for t in TARGETS:
    X,y,scaf=feat[t]; tr,te=scaffold_split(scaf,len(y),0); tr=np.array(tr); np.random.RandomState(0).shuffle(tr)
    for frac in [0.15,0.3,0.5,0.75,1.0]:
        k=max(40,int(len(tr)*frac)); print(f"   {t:12s} n_train={k:5d} R2={r2_score(y[te],boost().fit(X[tr[:k]],y[tr[:k]]).predict(X[te])):.3f}")
print("4) READ-ACROSS (1-NN Tanimoto)")
for t in TARGETS:
    X,y,scaf=feat[t]; B=(X[:,:2048]>0).astype(np.uint8); pc=B.sum(1); rs=[]
    for s in range(12):
        tr,te=scaffold_split(scaf,len(y),s); tr=np.array(tr)
        pred=[y[tr[int(np.argmax((B[tr]&B[i]).sum(1)/np.maximum(1,pc[tr]+pc[i]-(B[tr]&B[i]).sum(1))))]] for i in te]
        if len(np.unique(y[te]))>1: rs.append(r2_score(y[te],np.array(pred)))
    print(f"   {t:12s} read-across R2={np.mean(rs):.3f}")
